# CircuitSage Gemma LoRA

USER ACTION: run this notebook on Kaggle or Colab with a GPU. The pinned model id is `unsloth/gemma-3-4b-it`.

In [ ]:
!pip install -q "unsloth[colab-new]" trl datasets accelerate bitsandbytes

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastModel
import torch

MODEL_ID = "unsloth/gemma-3-4b-it"
DATASET_PATH = "train/dataset/circuitsage_qa.jsonl"
MAX_SEQ_LENGTH = 4096

In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)
model = FastModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

def format_example(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(format_example, remove_columns=dataset.column_names)

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        warmup_ratio = 0.03,
        logging_steps = 10,
        output_dir = "train/output/circuitsage-lora-trainer",
        optim = "adamw_8bit",
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
    ),
)
trainer.train()

In [ ]:
ADAPTER_DIR = "train/output/circuitsage-lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

In [ ]:
model.save_pretrained_gguf(
    "train/output",
    tokenizer,
    quantization_method = "q4_k_m",
)
# Rename the produced GGUF to train/output/circuitsage-lora-q4_k_m.gguf if Unsloth emits a model-specific filename.